In [1]:
import warnings
warnings.filterwarnings('ignore')
from network_analysis import NetworkAnalyzer

# 초기화
analyzer = NetworkAnalyzer()
results_summary = {}
save_path = '../result/'
data_path = '../db/processed_data/total_only.csv'

# 데이터 로드 및 준비
total_df_only = analyzer.preprocessor.load_and_prepare_data(data_path)
datasets = analyzer.preprocessor.create_group_datasets(total_df_only)

print("=== 한국인 식습관-건강 네트워크 분석 ===")
print(f"분석 대상: {len(total_df_only):,}명")
print(f"- 남성: {len(datasets['Men_All']):,}명")
print(f"- 여성: {len(datasets['Women_All']):,}명")
print(f"- 식품군: 12개, 건강지표: 23개\n")

# 기본 네트워크 생성
basic_networks = analyzer.create_basic_networks(datasets)
centrality_results = analyzer.centrality_analyzer.analyze_network_centrality_from_graphs(basic_networks)

# 그룹 분리
age_groups = ['under40', '40-49', '50-64', '65plus']
age_labels = ['40세 미만', '40-49세', '50-64세', '65세 이상']
health_types = ['diet_disease', 'diet_mets', 'diet_biomarker']
health_names = ['질병', 'MetS', '생체지표']

=== 한국인 식습관-건강 네트워크 분석 ===
분석 대상: 23,040명
- 남성: 12,294명
- 여성: 10,746명
- 식품군: 12개, 건강지표: 23개

Analyzing Total - Co-occurrence (poor)
Disconnected graph
Analyzing Total - Co-occurrence (non_poor)
Disconnected graph
Analyzing Total - Health Network (diet_disease)
Connected graph
Analyzing Total - Health Network (diet_mets)
Connected graph
Analyzing Total - Health Network (diet_biomarker)
Connected graph
Analyzing Men_All - Co-occurrence (poor)
Disconnected graph
Analyzing Men_All - Co-occurrence (non_poor)
Disconnected graph
Analyzing Men_All - Health Network (diet_disease)
Connected graph
Analyzing Men_All - Health Network (diet_mets)
Connected graph
Analyzing Men_All - Health Network (diet_biomarker)
Disconnected graph
Analyzing Men_under40 - Co-occurrence (poor)
Disconnected graph
Analyzing Men_under40 - Co-occurrence (non_poor)
Disconnected graph
Analyzing Men_under40 - Health Network (diet_disease)
Disconnected graph
Analyzing Men_under40 - Health Network (diet_mets)
Disconnected gra

## 2.1 전체 인구집단 분석

In [2]:
health_types = ['diet_disease', 'diet_mets', 'diet_biomarker']
health_names = ['질병', 'MetS', '생체지표']

for h_type, h_name in zip(health_types, health_names):
    print(f"\n◆ {h_name} 네트워크:")
    health_data = centrality_results['Total']['health'].get(h_type, {})
    
    health_nodes = [n for n in health_data['degree_centrality'].keys()
                    if n not in analyzer.preprocessor.diet_cols]
    
    # 연결 중심성(허브)
    health_degree = {n: health_data['degree_centrality'][n] for n in health_nodes}
    top_degree = sorted(health_degree.items(), key=lambda x: x[1], reverse=True)[:3]
    for node, score in top_degree:
        print(f"- {node}: {score:.3f}")


◆ 질병 네트워크:
- Obese: 0.667
- HL: 0.611
- DM: 0.556

◆ MetS 네트워크:
- △ WC: 0.750
- △ BP: 0.625
- IFG: 0.625

◆ 생체지표 네트워크:
- Weight: 0.450
- eGFR: 0.450
- BMI: 0.400


## 2.2 성별 간 분석

In [3]:
network_types = ['health', 'cooccurrence']
health_types = ['diet_disease', 'diet_mets', 'diet_biomarker']
health_names = ['질병', 'MetS', '생체지표']
cooccurrence_types = ['poor', 'non_poor']
cooccurrence_names = ['Poor Diet', 'Non-Poor Diet']
genders = ['Men_All', 'Women_All']

gender_comparison = {gender: {} for gender in genders}

for gender in genders:
    for h_type, h_name in zip(health_types, health_names):
        props = centrality_results[gender]['health'].get(h_type, {}).get('network_properties', {})
        gender_comparison[gender][h_name] = {
            'density': props.get('density', 0),
            'clustering': props.get('clustering', 0),
            'nodes': props.get('nodes', 0),
            'edges': props.get('edges', 0),
            'type': '건강'
        }
    for c_type, c_name in zip(cooccurrence_types, cooccurrence_names):
        props = centrality_results[gender]['cooccurrence'].get(c_type, {}).get('network_properties', {})
        gender_comparison[gender][c_name] = {
            'density': props.get('density', 0),
            'clustering': props.get('clustering', 0),
            'nodes': props.get('nodes', 0),
            'edges': props.get('edges', 0),
            'type': '식습관'
        }

print("\n◆ 네트워크 밀도 비교:")
for net_name in health_names + cooccurrence_names:
    men_density = gender_comparison['Men_All'].get(net_name, {}).get('density', 0)
    women_density = gender_comparison['Women_All'].get(net_name, {}).get('density', 0)
    print(f"  {net_name}: 남성={men_density:.3f}, 여성={women_density:.3f}")

# Fisher's Z & Cohen's d 분석
advanced_gender_df = analyzer.centrality_analyzer.calculate_fishers_z_and_cohens_d(centrality_results)
significant_diffs = advanced_gender_df[advanced_gender_df['Significant'] == True]

if not significant_diffs.empty:
    print("\n◆ 통계적으로 유의한 성별 차이 (p<0.05):")
    for _, row in significant_diffs.iterrows():
        print(f"  - {row['Node']} ({row['Network_Type']})")
        print(f"    남성={row['Men_Centrality']:.3f}, 여성={row['Women_Centrality']:.3f}")
        print(f"    Cohen's d={row['Cohens_D']:.2f} ({row['Effect_Size']})")


◆ 네트워크 밀도 비교:
  질병: 남성=0.345, 여성=0.263
  MetS: 남성=0.331, 여성=0.272
  생체지표: 남성=0.243, 여성=0.129
  Poor Diet: 남성=0.212, 여성=0.212
  Non-Poor Diet: 남성=0.212, 여성=0.212

◆ 통계적으로 유의한 성별 차이 (p<0.05):
  - Protein (diet_disease)
    남성=0.222, 여성=0.056
    Cohen's d=1.98 (Large)
  - Salty
Food (diet_disease)
    남성=0.222, 여성=0.056
    Cohen's d=1.98 (Large)
  - Stroke (diet_disease)
    남성=0.278, 여성=0.056
    Cohen's d=1.98 (Large)
  - Protein (diet_mets)
    남성=0.250, 여성=0.062
    Cohen's d=1.98 (Large)
  - Dairy (diet_mets)
    남성=0.062, 여성=0.188
    Cohen's d=-1.97 (Large)
  - TG (diet_biomarker)
    남성=0.400, 여성=0.000
    Cohen's d=1.99 (Large)
  - SBP (diet_biomarker)
    남성=0.050, 여성=0.300
    Cohen's d=-1.98 (Large)
  - High Fat
Meat (diet_biomarker)
    남성=0.300, 여성=0.100
    Cohen's d=1.98 (Large)
  - Protein (diet_biomarker)
    남성=0.100, 여성=0.000
    Cohen's d=1.96 (Large)
  - Weight (diet_biomarker)
    남성=0.500, 여성=0.100
    Cohen's d=1.99 (Large)
  - Grain (diet_biomarker)
    남성=0.1

## 2.3 성별/연령별 분석

In [4]:
print("\n◆ 연령별 네트워크 밀도 변화:")
age_density_data = {}

for gender in ['Men', 'Women']:
    gender_data = {}
    print(f"\n  {gender}성:")
    
    for h_type, h_name in zip(health_types, health_names):
        densities = []
        for age, label in zip(age_groups, age_labels):
            key = f"{gender}_{age}"
            if key in centrality_results:
                props = centrality_results[key]['health'].get(h_type, {}).get('network_properties', {})
                density = props.get('density', 0)
                densities.append(density)
            else:
                densities.append(0)
        
        gender_data[h_name] = densities
        print(f"    {h_name}: {' → '.join([f'{d:.3f}' for d in densities])}")
    
    age_density_data[gender] = gender_data


◆ 연령별 네트워크 밀도 변화:

  Men성:
    질병: 0.070 → 0.111 → 0.158 → 0.058
    MetS: 0.118 → 0.184 → 0.250 → 0.059
    생체지표: 0.171 → 0.190 → 0.181 → 0.124

  Women성:
    질병: 0.035 → 0.053 → 0.123 → 0.094
    MetS: 0.074 → 0.154 → 0.103 → 0.044
    생체지표: 0.100 → 0.138 → 0.105 → 0.052


In [5]:
print("\n◆ 연령별 건강지표 허브 변화:")

for h_type, h_name in zip(health_types, health_names):
    print(f"\n  {h_name} 네트워크:")
    
    for gender in ['Men', 'Women']:
        print(f"    {gender}성 상위 허브:")
        
        for age, label in zip(age_groups, age_labels):
            key = f"{gender}_{age}"
            if key in centrality_results:
                health_data = centrality_results[key]['health'].get(h_type, {})
                if 'degree_centrality' in health_data:
                    # 건강지표만 추출 (diet_cols 제외)
                    health_nodes = {n: v for n, v in health_data['degree_centrality'].items() 
                                  if n not in analyzer.preprocessor.diet_cols}
                    
                    if health_nodes:
                        top_health = max(health_nodes.items(), key=lambda x: x[1])
                        print(f"      {label}: {top_health[0]}({top_health[1]:.3f})")
                    else:
                        print(f"      {label}: 연결 없음")
                else:
                    print(f"      {label}: 데이터 없음")
            else:
                print(f"      {label}: 분석 대상 없음")



◆ 연령별 건강지표 허브 변화:

  질병 네트워크:
    Men성 상위 허브:
      40세 미만: Obese(0.278)
      40-49세: Obese(0.500)
      50-64세: Obese(0.500)
      65세 이상: DM(0.222)
    Women성 상위 허브:
      40세 미만: Obese(0.222)
      40-49세: Obese(0.333)
      50-64세: HTN(0.389)
      65세 이상: DM(0.278)

  MetS 네트워크:
    Men성 상위 허브:
      40세 미만: △ WC(0.375)
      40-49세: △ TG(0.500)
      50-64세: △ WC(0.625)
      65세 이상: △ WC(0.188)
    Women성 상위 허브:
      40세 미만: △ WC(0.188)
      40-49세: ▽ HDL-C(0.438)
      50-64세: △ WC(0.375)
      65세 이상: IFG(0.125)

  생체지표 네트워크:
    Men성 상위 허브:
      40세 미만: Weight(0.300)
      40-49세: Weight(0.400)
      50-64세: Weight(0.400)
      65세 이상: Weight(0.250)
    Women성 상위 허브:
      40세 미만: WC(0.200)
      40-49세: BMI(0.300)
      50-64세: Weight(0.300)
      65세 이상: Weight(0.100)


In [6]:
print("\n◆ 생애주기별 네트워크 변화 트렌드:")
lifecycle_df = analyzer.centrality_analyzer.analyze_lifecycle_network_evolution(centrality_results)

significant_trends = lifecycle_df[
    (abs(lifecycle_df['Change_Rate_Percent']) >= 20) | 
    (abs(lifecycle_df['Age_Correlation']) >= 0.5)
]

for gender in ['Men', 'Women']:
    gender_trends = significant_trends[significant_trends['Gender'] == gender]
    if not gender_trends.empty:
        print(f"\n  {gender}성 주요 변화:")
        
        for _, row in gender_trends.iterrows():
            direction = "증가" if row['Trend_Direction'] == 'Increasing' else "감소" if row['Trend_Direction'] == 'Decreasing' else "안정"
            print(f"    • {row['Network_Type']} {row['Metric']}: {direction} ({row['Change_Rate_Percent']:.1f}%)")



◆ 생애주기별 네트워크 변화 트렌드:

  Men성 주요 변화:
    • diet_mets density: 감소 (-49.6%)
    • diet_mets avg_degree_centrality: 감소 (-49.6%)
    • diet_biomarker density: 감소 (-27.6%)
    • diet_biomarker avg_degree_centrality: 감소 (-27.6%)

  Women성 주요 변화:
    • diet_disease density: 증가 (162.0%)
    • diet_disease avg_degree_centrality: 증가 (162.0%)
    • diet_mets density: 감소 (-39.5%)
    • diet_mets avg_degree_centrality: 감소 (-39.5%)
    • diet_biomarker density: 감소 (-47.1%)
    • diet_biomarker avg_degree_centrality: 감소 (-47.1%)


In [7]:
print("\n◆ 연령대별 우선 관리 대상 정리:")

priority_analysis = {}
for gender in ['Men', 'Women']:
    print(f"\n  {gender}성:")
    gender_priorities = {}
    
    for age, label in zip(age_groups, age_labels):
        priorities = []
        
        # Poor Diet 최고 허브
        if gender in age_hubs and age in age_hubs[gender] and age_hubs[gender][age]:
            top_poor_hub = age_hubs[gender][age][0][0]  # 최고 중심성 식품군
            priorities.append(f"식품군: {top_poor_hub}")
        
        # 질병 네트워크 최고 허브
        key = f"{gender}_{age}"
        if key in centrality_results:
            disease_data = centrality_results[key]['health'].get('diet_disease', {})
            if 'degree_centrality' in disease_data:
                health_nodes = {n: v for n, v in disease_data['degree_centrality'].items() 
                              if n not in analyzer.preprocessor.diet_cols}
                if health_nodes:
                    top_disease = max(health_nodes.items(), key=lambda x: x[1])
                    priorities.append(f"질병: {top_disease[0]}")
        
        gender_priorities[age] = priorities
        if priorities:
            print(f"    {label}: {', '.join(priorities)}")
        else:
            print(f"    {label}: 특이사항 없음")
    
    priority_analysis[gender] = gender_priorities

# 결과 저장을 위한 데이터 정리
age_analysis_results = {
    'age_density': age_density_data,
    'age_hubs': age_hubs,
    'lifecycle_trends': lifecycle_df,
    'priority_analysis': priority_analysis
}


◆ 연령대별 우선 관리 대상 정리:

  Men성:


NameError: name 'age_hubs' is not defined

# 3. 결과 저장

In [ ]:
results_summary['gender_comparison'] = gender_comparison
results_summary['lifecycle_evolution'] = lifecycle_df
results_summary['advanced_gender'] = advanced_gender_df

lifecycle_df.to_csv(f'{save_path}lifecycle_evolution.csv', index=False)
advanced_gender_df.to_csv(f'{save_path}gender_comparison.csv', index=False)

In [8]:
# 네트워크 시각화
analyzer.create_optimized_plots(basic_networks, centrality_results, save_path)

Created Men diet age progression plot
Created Women diet age progression plot
Created Men health-diet age progression plot
Created Women health-diet age progression plot
